# Adaptive Watermark Forgery Detection - Validation

This notebook validates the instance-based forgery detection approach:

1. **Z-score variance analysis**: Measure σ of regenerations from same prefix
2. **Attack detection**: Test detection rate on instance-based attacks
3. **False positive rate**: Ensure genuine samples aren't flagged
4. **Hyperparameter optimization**: Find optimal k (prefix length) and n (num regenerations)

In [ ]:
import json
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from tqdm import tqdm

# Add parent directory to path
sys.path.append('..')

from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig

## Configuration

In [ ]:
# Watermark configuration
WATERMARK_SCHEME = "KGW"
WATERMARK_CONFIG = "../config/KGW.json"
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
DEVICE = "auto"

# Experiment parameters
K_VALUES = [20, 30, 40, 50]  # Prefix lengths to test
N_VALUES = [30, 40, 50]       # Number of regenerations to test
THRESHOLD_SIGMA = 3.0         # Statistical threshold (3σ)

# Data paths
GENUINE_DATA = "../data/mistral-7b-v0.1_kgw_watermarked.jsonl"
ATTACKED_DATA = "../results/attacks/mistral-7b-v0.1_kgw_watermarked/paraphrase.jsonl"

## Setup Watermark Detector

In [ ]:
def setup_watermark():
    """Initialize watermark for generation and detection."""
    transformers_config = TransformersConfig(
        model=MODEL_NAME,
        device=DEVICE,
        max_new_tokens=200,
        min_length=0,
        do_sample=True,
        no_repeat_ngram_size=4
    )
    
    watermark = AutoWatermark.load(
        algorithm_name=WATERMARK_SCHEME,
        algorithm_config=WATERMARK_CONFIG,
        transformers_config=transformers_config
    )
    
    return watermark

print("Loading watermark detector...")
watermark = setup_watermark()
print("Watermark detector loaded!")

## Core Detection Algorithm

In [ ]:
def extract_prefix(text: str, k: int) -> str:
    """Extract first k tokens as prefix."""
    tokenizer = watermark.config.generation_tokenizer
    tokens = tokenizer.encode(text, add_special_tokens=False)[:k]
    prefix = tokenizer.decode(tokens)
    return prefix

def compute_z_score(text: str) -> float:
    """Compute z-score for text using watermark detector."""
    detection_result = watermark.detect_watermark(text, return_dict=True)
    return detection_result.get('score', 0.0)

def regenerate_samples(prefix: str, n: int) -> List[str]:
    """Regenerate n samples from prefix."""
    samples = []
    for _ in range(n):
        try:
            sample = watermark.generate_watermarked_text(prefix)
            samples.append(sample)
        except Exception as e:
            print(f"Generation failed: {e}")
            continue
    return samples

def detect_instance_forgery(text: str, k: int, n: int, threshold_sigma: float = 3.0) -> Tuple[bool, Dict]:
    """
    Detect if text is an instance-based forgery.
    
    Args:
        text: Suspicious text to evaluate
        k: Prefix length (number of tokens)
        n: Number of regenerations
        threshold_sigma: Statistical threshold (default 3σ)
    
    Returns:
        is_forged: Boolean indicating if text is likely forged
        stats: Dictionary with detection statistics
    """
    # 1. Extract prefix
    prefix = extract_prefix(text, k)
    
    # 2. Regenerate n samples
    regenerated_samples = regenerate_samples(prefix, n)
    
    if len(regenerated_samples) < n // 2:
        return False, {'error': 'Insufficient regenerations'}
    
    # 3. Compute z-scores
    suspicious_z = compute_z_score(text)
    legitimate_z_scores = [compute_z_score(s) for s in regenerated_samples]
    
    # 4. Statistical test
    μ = np.mean(legitimate_z_scores)
    σ = np.std(legitimate_z_scores)
    
    deviation = abs(suspicious_z - μ)
    is_forged = deviation > threshold_sigma * σ
    
    stats = {
        'suspicious_z': suspicious_z,
        'mean_z': μ,
        'std_z': σ,
        'deviation': deviation,
        'threshold': threshold_sigma * σ,
        'is_forged': is_forged,
        'num_regenerations': len(regenerated_samples),
        'all_z_scores': legitimate_z_scores
    }
    
    return is_forged, stats

## Experiment 1: Z-Score Variance Analysis

**Goal**: Confirm that regenerations from same prefix have tight distribution (σ < 1.0)

In [ ]:
# Load a sample of genuine watermarked texts
def load_jsonl(filepath: str, max_samples: int = None) -> List[Dict]:
    """Load JSONL file."""
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        print("Please generate watermarked samples first using: python -m generate.generator config/generate_config.json")
        return []
    
    samples = []
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            if line.strip():
                samples.append(json.loads(line))
    return samples

# Load genuine samples
genuine_samples = load_jsonl(GENUINE_DATA, max_samples=10)
print(f"Loaded {len(genuine_samples)} genuine samples")

In [ ]:
# Measure z-score variance for different k values
variance_results = {}

for k in K_VALUES:
    print(f"\nTesting k={k}...")
    all_stds = []
    
    for i, sample in enumerate(genuine_samples[:5]):  # Test on subset
        text = sample['full_text']
        
        # Extract prefix and regenerate
        prefix = extract_prefix(text, k)
        regenerated = regenerate_samples(prefix, n=30)
        
        # Compute z-scores
        z_scores = [compute_z_score(s) for s in regenerated]
        σ = np.std(z_scores)
        all_stds.append(σ)
        
        print(f"  Sample {i+1}: σ = {σ:.3f}")
    
    avg_std = np.mean(all_stds)
    variance_results[k] = {'stds': all_stds, 'avg_std': avg_std}
    print(f"  Average σ for k={k}: {avg_std:.3f}")

# Plot results
plt.figure(figsize=(10, 6))
k_vals = list(variance_results.keys())
avg_stds = [variance_results[k]['avg_std'] for k in k_vals]

plt.bar(k_vals, avg_stds, color='steelblue', alpha=0.7)
plt.axhline(y=1.0, color='r', linestyle='--', label='Target threshold (σ = 1.0)')
plt.xlabel('Prefix Length (k tokens)')
plt.ylabel('Average Z-Score Std Dev (σ)')
plt.title('Z-Score Variance vs Prefix Length')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('z_score_variance.png', dpi=150)
plt.show()

print(f"\n✓ Goal: σ < 1.0")
print(f"Result: Best k={min(variance_results, key=lambda k: variance_results[k]['avg_std'])} with σ={min(avg_stds):.3f}")

## Experiment 2: Attack Detection Rate

**Goal**: Detect instance-based attacks with >95% accuracy

In [ ]:
# Load attacked samples
attacked_samples = load_jsonl(ATTACKED_DATA, max_samples=20)
print(f"Loaded {len(attacked_samples)} attacked samples")

if len(attacked_samples) == 0:
    print("\nNo attacked samples found. Please generate attacks first using:")
    print("export OPENAI_API_KEY='your-key'")
    print("python -m attacks.attacker config/attack_config.json")

In [ ]:
# Test detection on attacked samples
detection_results = {}

for k in [30]:  # Use best k from variance analysis
    for n in [30, 40, 50]:
        print(f"\nTesting k={k}, n={n}...")
        
        detected = 0
        total = 0
        
        for i, sample in enumerate(tqdm(attacked_samples[:10], desc=f"k={k}, n={n}")):
            attacked_text = sample.get('attacked_response', '')
            if not attacked_text:
                continue
            
            is_forged, stats = detect_instance_forgery(attacked_text, k, n, THRESHOLD_SIGMA)
            
            if is_forged:
                detected += 1
            total += 1
        
        detection_rate = (detected / total * 100) if total > 0 else 0
        detection_results[(k, n)] = {
            'detected': detected,
            'total': total,
            'rate': detection_rate
        }
        
        print(f"  Detection rate: {detection_rate:.1f}% ({detected}/{total})")

print(f"\n✓ Goal: Detection rate > 95%")
best_config = max(detection_results, key=lambda x: detection_results[x]['rate'])
best_rate = detection_results[best_config]['rate']
print(f"Result: Best config k={best_config[0]}, n={best_config[1]} with {best_rate:.1f}% detection")

## Experiment 3: False Positive Rate

**Goal**: Ensure genuine samples aren't flagged (FPR < 1%)

In [ ]:
# Test on genuine watermarked samples
k_best = 30  # Use best k from previous experiments
n_best = 40  # Use best n from previous experiments

print(f"Testing FPR with k={k_best}, n={n_best}...")

false_positives = 0
total = 0

for i, sample in enumerate(tqdm(genuine_samples[:20], desc="Testing genuine samples")):
    text = sample['full_text']
    
    is_forged, stats = detect_instance_forgery(text, k_best, n_best, THRESHOLD_SIGMA)
    
    if is_forged:
        false_positives += 1
        print(f"\nFalse positive on sample {i+1}:")
        print(f"  Z-score: {stats['suspicious_z']:.3f}")
        print(f"  Mean: {stats['mean_z']:.3f}, Std: {stats['std_z']:.3f}")
        print(f"  Deviation: {stats['deviation']:.3f} > Threshold: {stats['threshold']:.3f}")
    
    total += 1

fpr = (false_positives / total * 100) if total > 0 else 0

print(f"\n✓ Goal: FPR < 1%")
print(f"Result: FPR = {fpr:.2f}% ({false_positives}/{total} false positives)")

if fpr < 1.0:
    print("✓ PASSED: False positive rate within acceptable threshold!")
else:
    print("✗ FAILED: False positive rate too high. Consider adjusting threshold_sigma.")

## Experiment 4: Z-Score Distribution Visualization

Compare distributions between genuine and forged texts

In [ ]:
# Collect z-scores for visualization
genuine_z_scores = []
forged_z_scores = []

print("Collecting z-scores for visualization...")

# Genuine samples
for sample in tqdm(genuine_samples[:10], desc="Genuine"):
    z = compute_z_score(sample['full_text'])
    genuine_z_scores.append(z)

# Forged samples
for sample in tqdm(attacked_samples[:10], desc="Forged"):
    attacked_text = sample.get('attacked_response', '')
    if attacked_text:
        z = compute_z_score(attacked_text)
        forged_z_scores.append(z)

# Plot distributions
plt.figure(figsize=(12, 6))

plt.hist(genuine_z_scores, bins=20, alpha=0.6, label='Genuine', color='green')
plt.hist(forged_z_scores, bins=20, alpha=0.6, label='Forged', color='red')

plt.axvline(x=np.mean(genuine_z_scores), color='green', linestyle='--', 
            label=f'Genuine Mean: {np.mean(genuine_z_scores):.2f}')
plt.axvline(x=np.mean(forged_z_scores), color='red', linestyle='--',
            label=f'Forged Mean: {np.mean(forged_z_scores):.2f}')

plt.xlabel('Z-Score')
plt.ylabel('Frequency')
plt.title('Z-Score Distribution: Genuine vs Forged Texts')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('z_score_distribution.png', dpi=150)
plt.show()

print(f"\nGenuine samples: μ={np.mean(genuine_z_scores):.3f}, σ={np.std(genuine_z_scores):.3f}")
print(f"Forged samples: μ={np.mean(forged_z_scores):.3f}, σ={np.std(forged_z_scores):.3f}")

## Summary of Results

In [ ]:
print("="*60)
print("VALIDATION SUMMARY")
print("="*60)

print("\n1. Z-Score Variance (Goal: σ < 1.0)")
if variance_results:
    best_k = min(variance_results, key=lambda k: variance_results[k]['avg_std'])
    best_std = variance_results[best_k]['avg_std']
    print(f"   ✓ Best k={best_k} with σ={best_std:.3f}")
    print(f"   {'PASSED' if best_std < 1.0 else 'NEEDS IMPROVEMENT'}")

print("\n2. Attack Detection Rate (Goal: > 95%)")
if detection_results:
    best_config = max(detection_results, key=lambda x: detection_results[x]['rate'])
    best_rate = detection_results[best_config]['rate']
    print(f"   ✓ Best config k={best_config[0]}, n={best_config[1]}")
    print(f"   Detection rate: {best_rate:.1f}%")
    print(f"   {'PASSED' if best_rate > 95 else 'NEEDS IMPROVEMENT'}")

print("\n3. False Positive Rate (Goal: < 1%)")
if 'fpr' in locals():
    print(f"   FPR: {fpr:.2f}%")
    print(f"   {'✓ PASSED' if fpr < 1.0 else '✗ NEEDS IMPROVEMENT'}")

print("\n4. Optimal Hyperparameters")
print(f"   Recommended: k={best_k if 'best_k' in locals() else 30}, n={40}")

print("\n" + "="*60)